# Ensemble weight sweep — logistic ↔ GBT

`04_model_predictions` suggested the served **50/50** logistic+GBT blend is dragged down by the weaker logistic. This finds the *best* blend weight **honestly**: three chronological splits — **train** (fit the two learners), **validation** (pick the weight), **test** (report unbiased). Picking the weight on the same data we score on would overfit; this doesn't.

Blend: `p = w·p_gbt + (1−w)·p_logistic`. w=0 → logistic only, w=1 → GBT only, w=0.5 → the current production ensemble.

In [ ]:
import sys
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style='whitegrid')
ROOT = Path.cwd()
while not (ROOT/'src'/'sportsball').exists() and ROOT != ROOT.parent: ROOT = ROOT.parent
sys.path.insert(0, str(ROOT/'src')); sys.path.insert(0, str(ROOT/'scripts'))
from train_eval_duckdb import load_events, build_market_pit, _pipeline, K, HFA, DEFAULT_DB
from sportsball.pipelines._elo import walk_forward
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import log_loss, accuracy_score

In [ ]:
raw = load_events(str(DEFAULT_DB))
results = [(d,h,a,hs,as_) for (d,h,a,hs,as_,_hc,_ac) in raw]
frows,_ = walk_forward(results, K, HFA, mov_enabled=True, carry=0.75, gap_days=90,
                       form_window=10, market_pit=build_market_pit(raw))
X = np.array([r.features for r in frows]); y = np.array([1 if r.actual>=1 else 0 for r in frows])
n = len(X); c1, c2 = int(0.70*n), int(0.85*n)   # train / val / test (chronological)
Xtr,ytr = X[:c1],y[:c1]; Xva,yva = X[c1:c2],y[c1:c2]; Xte,yte = X[c2:],y[c2:]
print(f'{n} games -> train {len(ytr)} | val {len(yva)} | test {len(yte)}')

In [ ]:
logit = _pipeline().fit(Xtr,ytr)
gbt = HistGradientBoostingClassifier(max_iter=200,learning_rate=0.05,max_depth=3,random_state=0).fit(Xtr,ytr)
va_l, va_g = logit.predict_proba(Xva)[:,1], gbt.predict_proba(Xva)[:,1]
te_l, te_g = logit.predict_proba(Xte)[:,1], gbt.predict_proba(Xte)[:,1]
def blend(pl,pg,w): return w*pg + (1-w)*pl
ws = np.round(np.arange(0,1.001,0.05),2)
val = pd.DataFrame({'w':ws,
  'val_logloss':[log_loss(yva,blend(va_l,va_g,w),labels=[0,1]) for w in ws],
  'val_acc':[accuracy_score(yva,(blend(va_l,va_g,w)>=0.5).astype(int)) for w in ws]})
w_star = float(val.loc[val.val_logloss.idxmin(),'w'])   # pick weight by validation log-loss
print(f'validation-optimal weight w* = {w_star} (GBT share)')

## Validation curve — pick the weight here

In [ ]:
fig, ax1 = plt.subplots(figsize=(9,5)); ax2 = ax1.twinx()
ax1.plot(val.w, val.val_logloss, 'o-', color='crimson', label='log-loss (lower=better)')
ax2.plot(val.w, val.val_acc, 's--', color='steelblue', label='accuracy')
ax1.axvline(w_star, color='green', lw=1.2, label=f'w*={w_star}')
ax1.axvline(0.5, color='gray', lw=1, ls=':', label='current 50/50')
ax1.set_xlabel('w = GBT weight (0=logistic only, 1=GBT only)')
ax1.set_ylabel('validation log-loss', color='crimson'); ax2.set_ylabel('validation accuracy', color='steelblue')
ax1.set_title('Blend weight sweep (validation)'); ax1.legend(loc='upper center'); plt.show()

## Unbiased test results

Apply the validation-chosen `w*` to the **untouched test set**, vs the three reference blends.

In [ ]:
def test_row(name,w):
    p = blend(te_l,te_g,w)
    return {'blend':name,'w(GBT)':w,'test_acc':round(accuracy_score(yte,(p>=0.5).astype(int)),4),
            'test_logloss':round(log_loss(yte,p,labels=[0,1]),4)}
rows = [test_row('logistic only',0.0), test_row('current 50/50',0.5),
        test_row('GBT only',1.0), test_row(f'val-optimal w*={w_star}',w_star)]
res = pd.DataFrame(rows); res

In [ ]:
cur = res[res.blend=='current 50/50'].iloc[0]
opt = res[res['w(GBT)']==w_star].iloc[0]
print(f'current 50/50 : acc {cur.test_acc:.4f}  log-loss {cur.test_logloss:.4f}')
print(f'val-optimal   : acc {opt.test_acc:.4f}  log-loss {opt.test_logloss:.4f}  (w={w_star})')
print(f'delta         : acc {opt.test_acc-cur.test_acc:+.4f}  log-loss {opt.test_logloss-cur.test_logloss:+.4f}')
print('\n-> ' + ('re-weighting helps on test; consider updating pipelines/train.py.'
      if opt.test_logloss < cur.test_logloss else
      'no out-of-sample gain — the 50/50 blend is fine; do not chase the val optimum.'))